### Example Usage of the XGBoost Classifier with Stratified K-Fold Cross-Validation

This notebook contains a simple example of how to use the model developed for the competition dataset.

---
Import necessary libraries and modules

In [2]:
import pandas as pd
from pipeline.pipeline import FullPipeline
from models.model import XGBoostWrapper
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import duckdb

Load the data from the CSV file using DuckDB and convert it to a DataFrame

In [5]:
query = f"SELECT * FROM read_csv_auto('data/train.csv')"
train_data = duckdb.query(query).to_df()
train_data.drop(['ID_code'], axis=1, inplace=True)
x, y = train_data.drop(columns=["target"]), train_data["target"]

Initialize and train the pipeline with the XGBoost model

In [6]:
pipeline = FullPipeline(model=XGBoostWrapper())
pipeline.fit(x.values, y.values)

[XGBoostWrapper] Using scale_pos_weight=8.95


Predict on the test file

In [8]:
import warnings
warnings.filterwarnings("ignore")

df_test = pd.read_csv("data/test.csv")
X_test = df_test.drop(columns=["ID_code"])
y_test_proba = pipeline.model.predict_proba(pipeline.transform(X_test))[:, 1]

Show the predictions

In [9]:
pd.DataFrame({
    "ID_code": df_test["ID_code"],
    "target": (y_test_proba >= 0.6).astype(int)
}).head()

,ID_code,target
0,test_0,0
1,test_1,0
2,test_2,1
3,test_3,1
4,test_4,0
